# Dataset 2 – Oxford Parkinson's Disease Detection
**Implementation:** Vishal Hanuman  
**Algorithms:** k-NN (HW1) · Gaussian Naive Bayes (HW2) · Random Forest (HW3)  
**Evaluation:** Stratified 10-fold CV · Accuracy · Macro F1

In [ ]:
import sys, os, subprocess

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run(['git', 'clone', f'https://github.com/vishalhanuman14/classical-ml-benchmark.git'], check=True)
    os.chdir('classical-ml-benchmark')

if '.' not in sys.path:
    sys.path.insert(0, '.')


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from algorithms.vishal import knn, naive_bayes, random_forest, utils

df = pd.read_csv('data/parkinsons.csv')
print(df.head(2))
print(f"Shape: {df.shape}, Columns: {list(df.columns)}")


In [ ]:
# 'status' is the target (0=healthy, 1=Parkinson's); drop 'name' column
df = df.drop(columns=['name'], errors='ignore')
target_col = 'status'
X_raw = df.drop(columns=[target_col]).values.astype(float)
y     = df[target_col].values.astype(str)
X_norm, _ = utils.normalize(X_raw, X_raw)

print(f"Instances: {len(y)}, Features: {X_norm.shape[1]}, Classes: {dict(zip(*np.unique(y, return_counts=True)))}")


## k-NN – Hyperparameter Sweep (k)

In [ ]:
K_VALUES  = [1, 3, 5, 7, 11, 15, 21, 31]
knn_results = {}

for k in K_VALUES:
    def model_fn(Xtr, ytr, Xte, k=k):
        Xtr_n, Xte_n = utils.normalize(Xtr, Xte)
        return knn.predict(Xtr_n, ytr, Xte_n, k)

    acc, f1 = utils.cross_validate(model_fn, X_raw, y, pos_label='1')
    knn_results[k] = (acc, f1)
    print(f"k={k:>3}  acc={acc:.4f}  f1={f1:.4f}")


## Naive Bayes – Hyperparameter Sweep (none / log-variance smoothing)

In [ ]:
# NB has no major hyperparameter; sweep a small additive variance floor (epsilon)
EPSILONS  = [1e-9, 1e-7, 1e-5, 1e-3, 0.01, 0.1]
nb_results = {}

for eps in EPSILONS:
    def model_fn(Xtr, ytr, Xte, eps=eps):
        import algorithms.vishal.naive_bayes as nb_mod
        import importlib, types
        # Patch std floor for this run
        m = nb_mod.train(Xtr, ytr)
        for c in m['classes']:
            m['stds'][c] = np.maximum(m['stds'][c], eps)
        return nb_mod.predict(m, Xte)

    acc, f1 = utils.cross_validate(model_fn, X_norm, y, pos_label='1')
    nb_results[eps] = (acc, f1)
    print(f"eps={eps}  acc={acc:.4f}  f1={f1:.4f}")


## Random Forest – Hyperparameter Sweep (ntree)

In [ ]:
feature_types = ['num'] * X_norm.shape[1]
NTREE_VALUES  = [5, 10, 20, 30, 50, 75, 100, 150]
rf_results    = {}

for nt in NTREE_VALUES:
    def model_fn(Xtr, ytr, Xte, nt=nt):
        trees = random_forest.train(Xtr, ytr, feature_types, nt)
        return random_forest.predict(trees, Xte)

    acc, f1 = utils.cross_validate(model_fn, X_norm, y, pos_label='1')
    rf_results[nt] = (acc, f1)
    print(f"ntree={nt:>4}  acc={acc:.4f}  f1={f1:.4f}")


## Plots & Summary

In [ ]:
os.makedirs('results', exist_ok=True)

# k-NN curve
ks   = list(knn_results.keys())
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(ks, [knn_results[k][0] for k in ks], '-o', label='Accuracy')
ax.plot(ks, [knn_results[k][1] for k in ks], '-s', label='F1')
ax.set_xlabel('k'); ax.set_title("k-NN vs k  (Parkinson's)")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig('results/parkinsons_knn_k_sweep.png', dpi=150); plt.show()

# RF curve
nts = list(rf_results.keys())
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(nts, [rf_results[n][0] for n in nts], '-o', label='Accuracy')
ax.plot(nts, [rf_results[n][1] for n in nts], '-s', label='F1')
ax.set_xlabel('ntree'); ax.set_title("Random Forest vs ntree  (Parkinson's)")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig('results/parkinsons_rf_ntree_sweep.png', dpi=150); plt.show()


In [ ]:
best_k   = max(knn_results, key=lambda k: knn_results[k][1])
best_eps = max(nb_results,  key=lambda e: nb_results[e][1])
best_nt  = max(rf_results,  key=lambda n: rf_results[n][1])

print(f"{'Algorithm':<20} {'Best HP':<15} {'Accuracy':>10} {'F1 Score':>10}")
print('-' * 60)
print(f"{'k-NN':<20} {'k='+str(best_k):<15} {knn_results[best_k][0]:>10.4f} {knn_results[best_k][1]:>10.4f}")
print(f"{'Naive Bayes':<20} {'eps='+str(best_eps):<15} {nb_results[best_eps][0]:>10.4f} {nb_results[best_eps][1]:>10.4f}")
print(f"{'Random Forest':<20} {'ntree='+str(best_nt):<15} {rf_results[best_nt][0]:>10.4f} {rf_results[best_nt][1]:>10.4f}")
